pour tester les imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain_community")

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DataFrameLoader
from langchain_mistralai import MistralAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from mistralai.client import Mistral

import os
from dotenv import load_dotenv
import time
import numpy as np
from tqdm import tqdm

import pandas as pd

load_dotenv()

C:\Users\mdalm\AppData\Local\Temp\ipykernel_2908\4217399275.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

## Vectorisation des descriptions des événements de l'agenda

Test de l'api

In [ ]:
api_key = os.environ.get("MISTRAL_API_KEY")
model = "mistral-embed"

client = Mistral(api_key=api_key)

embeddings_batch_response = client.embeddings.create(
    model=model,
    inputs=["Embed this sentence.", "As well as this one."],
)

## Récupération des données

On appelle l'api public d'openAgenda sans api key pour récupérer les événements des Hauts-de-France en 2026

In [ ]:
import requests
headers = {
    "Content-Type": "application/json"
}
response = requests.get("https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records/?lang=fr&limit=10&offset=0&where=YEAR%28firstdate_begin%29+%3D+2026+AND+YEAR%28lastdate_begin%29+%3D+2026+AND+location_region+%3D%27Hauts-de-France%27", headers=headers)
print(f"{response}")

<Response [200]>


On transforme la réponse en DataFrame pour la suite du traitement

In [8]:
import json

print(f"{response.content}")
donnees = json.loads(response.content)
liste_resultats = donnees.get("results", [])
evenement_pd = pd.DataFrame(liste_resultats)

print(f"taille data : {len(evenement_pd)}")
evenement_pd.describe

b'{"total_count": 1, "results": [{"uid": "53403045", "slug": "nuit-internationale-de-la-chauve-souris-8784448", "canonicalurl": "https://openagenda.com/nuit-internationale-de-la-chauve-souris/events/nuit-internationale-de-la-chauve-souris-8784448", "title_fr": "Nuit Internationale de la Chauve-Souris", "description_fr": "Pr\\u00e9sentation en salle des Chiropt\\u00e8res pr\\u00e9sents en France et sur la commune. Poursuite de la soir\\u00e9e avec une balade nocturne dans le parc de la ville avec utilisation de Batbox.", "longdescription_fr": "<p>Rendez-vous \\u00e0 20h30 \\u00e0 l\\u2019H\\u00f4tel de Ville pour une premi\\u00e8re rencontre avec des sp\\u00e9cialistes qui vous feront d\\u00e9couvrir les diff\\u00e9rentes esp\\u00e8ces pr\\u00e9sentes sur le territoire, leur mode de vie, leur r\\u00f4le essentiel dans les \\u00e9cosyst\\u00e8mes, ainsi que les croyances et l\\u00e9gendes qui les entourent.<br>Pr\\u00e9sentes sur tous les continents, les chauves-souris occupent des milie

<bound method NDFrame.describe of         uid                                             slug  \
0  53403045  nuit-internationale-de-la-chauve-souris-8784448   

                                        canonicalurl  \
0  https://openagenda.com/nuit-internationale-de-...   

                                  title_fr  \
0  Nuit Internationale de la Chauve-Souris   

                                      description_fr  \
0  Présentation en salle des Chiroptères présents...   

                                  longdescription_fr  \
0  <p>Rendez-vous à 20h30 à l’Hôtel de Ville pour...   

                            conditions_fr  \
0  sur inscription, gratuit, grand public   

                                         keywords_fr  \
0  [découverte, Batbox, observation, balade noctu...   

                                               image imagecredits  ...  \
0  https://img.openagenda.com/main/7ea269e3aff544...         None  ...   

  originagenda_uid contributor_email contributor_con

Vectorisation des descriptions

In [ ]:
print(f"{evenement_pd.columns}")

evenement_pd = evenement_pd.dropna(subset=["Description"])

In [ ]:
descriptions_to_embed = evenement_pd.Description.to_list()

taille_du_lot = 100
tous_les_embeddings = []

# Exécution par paquets à cause des limitations de l'api
for i in tqdm(range(0, len(descriptions_to_embed), taille_du_lot), desc="Génération des vecteurs"):
    lot = descriptions_to_embed[i : i + taille_du_lot]
    
    reponse = client.embeddings.create(
        model=model,
        inputs=lot
    )
    
    for element in reponse.data:
        tous_les_embeddings.append(element.embedding)
        
    # Pause de sécurité pour ne pas dépasser les limites de l'API
    time.sleep(1)

# Insertion dans FAISS
vecteurs_numpy = np.array(tous_les_embeddings).astype('float32')
dimension = vecteurs_numpy.shape[1]

In [ ]:
df_metadata = evenement_pd.drop(columns=["Description"])
metadatas = df_metadata.to_dict(orient="records")

textes_et_vecteurs = list(zip(descriptions_to_embed, vecteurs_numpy))

embeddings = MistralAIEmbeddings(model="mistral-embed")

vector_store = FAISS.from_embeddings(
    text_embeddings=textes_et_vecteurs,
    embedding=embeddings,
    metadatas=metadatas
)

vector_store.save_local("mon_index_langchain_evenements")

print(f"✅ Base créée avec succès à partir de {vector_store.index.ntotal} vecteurs existants !")

Pour un index optimisé utilisons HNSW

In [27]:
import faiss
import uuid
from langchain_community.docstore.in_memory import InMemoryDocstore

# Configuration de l'algorithme HNSW
dimension = 1024  # Le modèle 'mistral-embed' génère des vecteurs de 1024 dimensions
liens_par_noeud = 32  # Paramètre HNSW

index_optimise = faiss.IndexHNSWFlat(dimension, liens_par_noeud)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index_optimise,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

textes_et_vecteurs = list(zip(descriptions_to_embed, vecteurs_numpy))

# LangChain demande un identifiant unique (ID) pour chaque document ajouté manuellement
ids = [str(uuid.uuid4()) for _ in range(len(descriptions_to_embed))]

vector_store.add_embeddings(
    text_embeddings=textes_et_vecteurs,
    metadatas=metadatas,
    ids=ids
)

# 4. Sauvegarde de la nouvelle base super-rapide
vector_store.save_local("mon_index_langchain_evenements_hnsw_rapide")
print("✅ Index HNSW optimisé et sauvegardé !")

✅ Index HNSW optimisé et sauvegardé !


Evaluons les deux index

In [28]:
import time

print("=== 1. CHARGEMENT ET TAILLE DES DEUX INDEX ===")

# Mesure du temps de chargement de l'index Flat L2
t0 = time.perf_counter()
db_flat = FAISS.load_local("mon_index_langchain_evenements", embeddings, allow_dangerous_deserialization=True)
t_load_flat = (time.perf_counter() - t0) * 1000
print(f"Index Flat L2 chargé en {t_load_flat:.2f} ms")

# Mesure du temps de chargement de l'index HNSW
t0 = time.perf_counter()
db_hnsw = FAISS.load_local("mon_index_langchain_evenements_hnsw_rapide", embeddings, allow_dangerous_deserialization=True)
t_load_hnsw = (time.perf_counter() - t0) * 1000
print(f"Index HNSW chargé en {t_load_hnsw:.2f} ms")

# Taille des fichiers sur disque
taille_flat_mo = os.path.getsize("mon_index_langchain_evenements/index.faiss") / (1024 * 1024)
taille_hnsw_mo = os.path.getsize("mon_index_langchain_evenements_hnsw_rapide/index.faiss") / (1024 * 1024)

print(f"📦 Index Flat L2 : {taille_flat_mo:.2f} Mo sur disque | {db_flat.index.ntotal} vecteurs | Chargement : {t_load_flat:.1f} ms")
print(f"⚡ Index HNSW    : {taille_hnsw_mo:.2f} Mo sur disque | {db_hnsw.index.ntotal} vecteurs | Chargement : {t_load_hnsw:.1f} ms")

=== 1. CHARGEMENT ET TAILLE DES DEUX INDEX ===
Index Flat L2 chargé en 560.69 ms
Index HNSW chargé en 469.86 ms
📦 Index Flat L2 : 96.50 Mo sur disque | 24705 vecteurs | Chargement : 560.7 ms
⚡ Index HNSW    : 102.92 Mo sur disque | 24705 vecteurs | Chargement : 469.9 ms


In [29]:
print("\n=== 2. BENCHMARK DES TEMPS D'EXÉCUTION (LATENCE DE RECHERCHE) ===")

nb_requetes = 50
k_voisins = 5
# Sélection d'un échantillon de vecteurs pour le test
vecteurs_test = vecteurs_numpy[:nb_requetes]

# Warmup du cache mémoire
db_flat.similarity_search_by_vector(vecteurs_test[0].tolist(), k=k_voisins)
db_hnsw.similarity_search_by_vector(vecteurs_test[0].tolist(), k=k_voisins)

# Mesure Flat L2
print(f"Lancement de {nb_requetes} recherches sur Flat L2 (k={k_voisins})...")
debut_flat = time.perf_counter()
for vec in vecteurs_test:
    _ = db_flat.similarity_search_by_vector(vec.tolist(), k=k_voisins)
duree_flat = time.perf_counter() - debut_flat
latence_flat_ms = (duree_flat / nb_requetes) * 1000
print(f"Flat L2 terminé : durée totale = {duree_flat*1000:.2f} ms | latence moyenne = {latence_flat_ms:.2f} ms/requête")

# Mesure HNSW
print(f"Lancement de {nb_requetes} recherches sur HNSW (k={k_voisins})...")
debut_hnsw = time.perf_counter()
for vec in vecteurs_test:
    _ = db_hnsw.similarity_search_by_vector(vec.tolist(), k=k_voisins)
duree_hnsw = time.perf_counter() - debut_hnsw
latence_hnsw_ms = (duree_hnsw / nb_requetes) * 1000
print(f"HNSW terminé : durée totale = {duree_hnsw*1000:.2f} ms | latence moyenne = {latence_hnsw_ms:.2f} ms/requête")

gain = latence_flat_ms / latence_hnsw_ms if latence_hnsw_ms > 0 else 1.0
print(f"⏱️ Résultat de la comparaison de latence sur {nb_requetes} requêtes :")
print(f"  • Flat L2 : {latence_flat_ms:.2f} ms / requête")
print(f"  • HNSW    : {latence_hnsw_ms:.2f} ms / requête")
print(f"  🚀 Facteur d'accélération HNSW : x{gain:.2f}")


=== 2. BENCHMARK DES TEMPS D'EXÉCUTION (LATENCE DE RECHERCHE) ===
Lancement de 50 recherches sur Flat L2 (k=5)...
Flat L2 terminé : durée totale = 819.50 ms | latence moyenne = 16.39 ms/requête
Lancement de 50 recherches sur HNSW (k=5)...
HNSW terminé : durée totale = 27.27 ms | latence moyenne = 0.55 ms/requête
⏱️ Résultat de la comparaison de latence sur 50 requêtes :
  • Flat L2 : 16.39 ms / requête
  • HNSW    : 0.55 ms / requête
  🚀 Facteur d'accélération HNSW : x30.05


In [30]:
print("\n=== 3. ÉVALUATION DE L'EFFICACITÉ SÉMANTIQUE (RECALL@K) ===")
# L'index Flat L2 calcule la recherche exacte (vérité terrain).
# On évalue quel pourcentage des k voisins exacts est retrouvé par HNSW.

k = 5
recalls = []

print(f"Calcul du Recall@{k} sur un échantillon de {nb_requetes} vecteurs...")
for vec in vecteurs_test:
    # Voisins exacts retournés par Flat L2
    res_flat = db_flat.similarity_search_by_vector(vec.tolist(), k=k)
    textes_flat = {doc.page_content for doc in res_flat}
    
    # Voisins approximatifs retournés par HNSW
    res_hnsw = db_hnsw.similarity_search_by_vector(vec.tolist(), k=k)
    textes_hnsw = {doc.page_content for doc in res_hnsw}
    
    # Taux de recouvrement
    intersection = len(textes_flat.intersection(textes_hnsw))
    recalls.append(intersection / k)

recall_moyen = np.mean(recalls) * 100
print(f"Recall@{k} moyen obtenu : {recall_moyen:.2f}%")
print(f"🎯 Taux de rappel (Recall@{k}) de l'index HNSW : {recall_moyen:.2f}%")
print("💡 Un taux proche de 100% démontre que l'approximation HNSW préserve la qualité des résultats tout en étant plus rapide.")


=== 3. ÉVALUATION DE L'EFFICACITÉ SÉMANTIQUE (RECALL@K) ===
Calcul du Recall@5 sur un échantillon de 50 vecteurs...
Recall@5 moyen obtenu : 53.20%
🎯 Taux de rappel (Recall@5) de l'index HNSW : 53.20%
💡 Un taux proche de 100% démontre que l'approximation HNSW préserve la qualité des résultats tout en étant plus rapide.


In [31]:
print("\n=== 4. RECHERCHE TEXTUELLE RÉELLE AVEC LOGGING DES TEMPS ===")

requete_test = "atelier d'initiation informatique et nouvelles technologies"
k_resultats = 3

print(f"Recherche textuelle pour : '{requete_test}'")

# Recherche avec Flat L2
t0 = time.perf_counter()
resultats_flat = db_flat.similarity_search_with_score(requete_test, k=k_resultats)
duree_texte_flat = (time.perf_counter() - t0) * 1000
print(f"Recherche Flat L2 exécutée en {duree_texte_flat:.2f} ms")

# Recherche avec HNSW
t0 = time.perf_counter()
resultats_hnsw = db_hnsw.similarity_search_with_score(requete_test, k=k_resultats)
duree_texte_hnsw = (time.perf_counter() - t0) * 1000
print(f"Recherche HNSW exécutée en {duree_texte_hnsw:.2f} ms")

print(f"\n🔎 Requête : \"{requete_test}\"")
print(f"\n[Flat L2 - {duree_texte_flat:.2f} ms]")
for i, (doc, score) in enumerate(resultats_flat, 1):
    print(f"  {i}. {doc.metadata.get('Titre')} (Distance: {score:.4f}) - {doc.metadata.get('Ville')}")

print(f"\n[HNSW - {duree_texte_hnsw:.2f} ms]")
for i, (doc, score) in enumerate(resultats_hnsw, 1):
    print(f"  {i}. {doc.metadata.get('Titre')} (Distance: {score:.4f}) - {doc.metadata.get('Ville')}")


=== 4. RECHERCHE TEXTUELLE RÉELLE AVEC LOGGING DES TEMPS ===
Recherche textuelle pour : 'atelier d'initiation informatique et nouvelles technologies'
Recherche Flat L2 exécutée en 329.53 ms
Recherche HNSW exécutée en 187.40 ms

🔎 Requête : "atelier d'initiation informatique et nouvelles technologies"

[Flat L2 - 329.53 ms]
  1. Initiation au thérémine - Charlotte Dubois (Distance: 0.3478) - Sars-Poteries
  2. "PRATIQUES NUMERIQUES" (Distance: 0.4018) - Villers-Cotterêts
  3. Les RDV numériques du vendredi – Vers l’autonomie (Distance: 0.4224) - Vervins

[HNSW - 187.40 ms]
  1. Initiation au thérémine - Charlotte Dubois (Distance: 0.3478) - Sars-Poteries
  2. "PRATIQUES NUMERIQUES" (Distance: 0.4018) - Villers-Cotterêts
  3. Les RDV numériques du vendredi – Vers l’autonomie (Distance: 0.4224) - Vervins


L'index HNSW gagne haut la main !